# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display required metadata
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", getattr(metadata, 'identifier', None))
print("License:", getattr(metadata, 'license', None))
print("Keywords:", getattr(metadata, 'keywords', None))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by their @id
print("Available Record Sets (by @id):")
record_sets = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
for rs in record_sets:
    print(f"  - {rs}")

# If at least one record set exists, list its fields by @id and @type
if record_sets:
    print("\nFields for first record set:")
    rs_meta = None
    for rsd in dataset.metadata.to_json().get('recordSet', []):
        if rsd['@id'] == record_sets[0]:
            rs_meta = rsd
            break
    if rs_meta is not None:
        fields = rs_meta.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            if isinstance(f, dict):
                print(f"  - {f.get('@id')}: type={f.get('@type')}")
            else:
                print(f"  - {f}")
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_sets = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
dataframes = {}

for record_set in record_sets:
    print(f"Loading records for record set: {record_set}")
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)
    print(f"  Columns: {dataframes[record_set].columns.tolist()}")

# Preview the first record set
if record_sets:
    main_rs = record_sets[0]
    print(dataframes[main_rs].head())
else:
    main_rs = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA: Select and analyze numeric fields (e.g., Molecular Weight, Peptide Count)

# Choose the relevant record set and field @ids. Adjust these based on data overview output.
record_set_id = main_rs  # For demonstration, use first RS
numeric_fields = []
if record_set_id and not dataframes[record_set_id].empty:
    # Heuristically find numeric columns
    df = dataframes[record_set_id]
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_fields.append(c)
    print("Numeric fields found:", numeric_fields)
else:
    print("No data to analyze.")

# Use the first numeric field for demonstration
if numeric_fields:
    numeric_field = numeric_fields[0]
    threshold = df[numeric_field].quantile(0.75)  # filter top 25% values

    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a (possibly categorical) column, if exists
    group_field = None
    for c in df.columns:
        if df[c].dtype == 'object' and c != numeric_field:
            group_field = c
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the main numeric field, if present
if main_rs and numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[main_rs][numeric_fields[0]], kde=True)
    plt.title(f"Distribution of {numeric_fields[0]}")
    plt.xlabel(numeric_fields[0])
    plt.ylabel("Frequency")
    plt.show()

    if len(numeric_fields) > 1:
        plt.figure(figsize=(7,7))
        sns.scatterplot(
            data=dataframes[main_rs],
            x=numeric_fields[0],
            y=numeric_fields[1]
        )
        plt.title(f"Scatterplot of {numeric_fields[0]} vs {numeric_fields[1]}")
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* With `mlcroissant`, we have loaded the FAIR^2 mass spectrometry dataset for human mast cell EV analysis directly from a Croissant schema URL.
* We explored available record sets and fields by their `@id`, loaded records into pandas DataFrames, and performed basic exploratory data analysis—including normalization and visualization of numeric attributes.
* To go further, explore meaningful relationships between protein properties, filter by experimental attributes, or visualize modifications reported in the dataset.

_For further documentation, see: https://mlcommons.github.io/croissant-python/_